# Fast/slow exposure relationship audit

This notebook uses the new `signal_relationship_audit.py` helper and the repository's existing `overlay_arms.py`. It does **not** force the correlation to 0.17.

In [1]:
from pathlib import Path
import pandas as pd

from signal_relationship_audit import (
    resolve_run,
    build_exact_signals,
    relationship_table,
    performance_table,
    audit_variants,
    plot_timeline,
    plot_full_scatter,
    plot_stress_scatter,
    plot_rolling_corr,
)


## 1. Canonical presentation run

Use `main_dyn_strong` first. Change `RUN_NAME` only when deliberately testing another archived result.

In [2]:
# RUN_NAME = 'main_dyn_strong'
RUN_NAME = 'advanced_smart_bayesian'


RUN = resolve_run(RUN_NAME)
OUT = RUN / 'signal_relationship_audit'
OUT.mkdir(parents=True, exist_ok=True)

arms, K = build_exact_signals(RUN, weights_col='weights', cvar_col='cvar_model')

print('rows:', len(K))
print('range:', K.index.min(), '->', K.index.max())
print('full-sample rho:', K['fast'].corr(K['slow']))


rows: 83
range: 2019-02-01 00:00:00 -> 2025-12-01 00:00:00
full-sample rho: 0.29652908919981696


## 2. Full sample and stress-period correlations

In [3]:
summary = relationship_table(K)
display(summary.round(4))

summary.to_csv(OUT / 'signal_relationship_summary.csv', index=False)


,period,n,pearson_r,pearson_p,pearson_ci_low,pearson_ci_high,spearman_rho,spearman_p,ols_slope_slow_on_fast,ols_intercept,mean_fast_exposure,mean_slow_exposure,mean_combined_exposure,share_any_timer_derisks,share_both_timers_derisk
0,full_sample,83,0.2965,0.0065,0.0864,0.4814,0.2490,0.0232,0.2519,0.6429,0.6416,0.8045,0.5450,0.8795,0.5663
1,2020,12,0.5426,0.0683,-0.0454,0.8514,0.2786,0.3805,0.3736,0.4246,0.6680,0.6742,0.5116,0.8333,0.6667
2,2022,12,-0.0018,0.9956,-0.5751,0.5727,0.1259,0.6967,-0.0012,0.6583,0.4126,0.6578,0.3120,1.0000,1.0000
3,stress_2020_2022,24,0.4537,0.0260,0.0616,0.7245,-0.0671,0.7553,0.2652,0.5227,0.5403,0.6660,0.4118,0.9167,0.8333
4,calm_remainder,59,0.1309,0.3229,-0.1295,0.3745,0.1169,0.3778,0.1147,0.7826,0.6827,0.8609,0.5992,0.8644,0.4576


## 3. Performance of base / fast / slow / combined arms

In [4]:
performance = performance_table(arms, K)
display(performance.round(4))

performance.to_csv(OUT / 'signal_performance_by_period.csv', index=False)


,period,arm,n_months,total_return_pct,annual_return_pct,annual_vol_pct,sharpe,max_drawdown_pct
0,full_sample,base,83,156.9571,14.6191,19.5908,0.6948,-15.3610
1,full_sample,fast,83,105.3213,10.9612,11.2833,0.8038,-10.6527
2,full_sample,slow,83,124.1020,12.3743,15.9072,0.6896,-15.1959
3,full_sample,combined,83,92.3554,9.9196,10.0676,0.7938,-9.8868
4,2020,base,12,17.4612,17.4612,18.4239,0.8561,-14.8341
5,2020,fast,12,8.2910,8.2910,12.8458,0.5263,-10.6527
6,2020,slow,12,10.2658,10.2658,12.8895,0.6663,-11.6219
7,2020,combined,12,5.6632,5.6632,10.5085,0.3845,-9.4142
8,2022,base,12,8.5329,8.5329,30.3113,0.3428,-14.7908
9,2022,fast,12,3.6809,3.6809,11.0778,0.1969,-4.8726


## 4. Presentation-style plots

In [5]:
plot_timeline(K, OUT / 'presentation_timer_timeline.png')
plot_full_scatter(K, OUT / 'presentation_timer_scatter.png')
plot_stress_scatter(K, OUT / 'presentation_timer_stress_scatter.png')
rolling = plot_rolling_corr(K, OUT / 'presentation_timer_rolling_corr.png', window=12)

print('saved plots to:', OUT)


saved plots to: /Users/mac/Downloads/Sequential-stochastic-portfolio-optimization-/results/advanced_smart_bayesian/signal_relationship_audit


## 5. Optional audit of archived construction variants

This is for diagnosis only. Do not select a variant merely because its correlation is closer to 0.17.

In [6]:
audit = audit_variants(RUN)
display(audit.round(4))
audit.to_csv(OUT / 'signal_variant_audit.csv', index=False)


,run,weights_col,cvar_col,n,corr_full_sample,corr_2020,corr_2022,corr_stress_2020_2022,corr_calm_remainder,abs_gap_to_presentation_0_17
0,advanced_smart_bayesian,weights,cvar_model,83,0.2965,0.5426,-0.0018,0.4537,0.1309,0.1265
1,advanced_smart_bayesian,weights,cvar_model_raw,83,0.2351,0.6470,0.6734,0.4016,0.0680,0.0651


## 6. Test a different run

For example, `main_dyn_on` may legitimately produce a different correlation from the presentation.

In [7]:
# TEST_RUN = resolve_run('main_dyn_on')
# _, K_test = build_exact_signals(TEST_RUN)
# display(relationship_table(K_test).round(4))
